tried to run it, but look at the original function instead  
C:\Users\Erik Liu\OneDrive - NTNU\PhD\Courses\UHI post processing\postprocess_uhi\mjosa_database\create_paper_plots\navigation.ipynb

In [5]:
import sys
import os

# Add path to eely_log_plot folder (it's in the same directory as this notebook)
sys.path.append(os.path.abspath("./eely_log_plot"))

from Eelume import PyPost as pp
import matplotlib.pyplot as plt
import datetime as datetime
import numpy as np

# Add path to x_gref4hsi_by_liu folder (contains utils)
sys.path.append(os.path.abspath("../../../../../x_gref4hsi_by_liu"))

from utils.analyze_log_file import LogData

plt.rcParams["figure.figsize"] = (3, 2)

import types

%matplotlib inline


ModuleNotFoundError: No module named 'utm'

In [2]:
filename = r"E:\mjosa\29\log_files\LOG_2024-10-29_10-13-32.db3"
db = pp.DatabaseHandler(filename)

start_time = datetime.datetime(2024, 10, 29, 10, 10, 00, tzinfo=datetime.timezone.utc)
end_time = datetime.datetime(2024, 10, 29, 13, 15, 00, tzinfo=datetime.timezone.utc)

# time comments:
# 10:10 we start to move, so good place to start
# 13:15 we start to assend, so can stop here

analyzer = pp.MotionAnalyzer(db, start_time=start_time, end_time=end_time)
csv_file_path = r"E:\mjosa_new\navigation_data\nav_data_from_logfile.csv"

In [3]:
main_csv = r"E:\mjosa_new\navigation_data\nav_data_from_logfile.csv"
log_data = LogData(main_csv)

# 2. Define Lake Mjøsa origin in degrees
origin_deg = (60.8011575, 10.7122345)

In [ ]:
# Add the missing deg_to_meter_simple method to LogData class
def deg_to_meter_simple(self, latitudes, longitudes, origin=None):
    """
    Convert lat/lon → metres.
    If `origin` is None, the first sample is used.
    Otherwise give (lat0, lon0) as a tuple.
    """
    import numpy as np

    lat0, lon0 = (latitudes[0], longitudes[0]) if origin is None else origin
    R = 6_378_137.0  # WGS-84 equatorial radius [m]

    lat_rad = np.radians(latitudes)
    lon_rad = np.radians(longitudes)
    lat0_rad = np.radians(lat0)
    lon0_rad = np.radians(lon0)

    east = (lon_rad - lon0_rad) * R * np.cos(lat0_rad)
    north = (lat_rad - lat0_rad) * R
    return east, north


# Apply the method to your LogData class
LogData.deg_to_meter_simple = deg_to_meter_simple

In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# Monkey-patch: plot_with_highlighted_transects
#   • main track (all data)      → black (2D), heat-colored by depth (3D)
#   • normal transects           → blue, no legend
#   • highlighted transect       → red, single legend entry
#   • legend drawn inside axes; no superfluous titles
# ─────────────────────────────────────────────────────────────────────────────
def plot_with_highlighted_transects(
    self,
    transect_csv_paths: list,
    highlighted_csv_paths: list = None,
    highlight_label: str = "Highlighted Transect",
    use_meters: bool = False,
    origin: tuple = None,
):
    import os
    import pandas as pd
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    import numpy as np

    # 0) Prepare main-track coords
    if use_meters:
        if origin is None:
            raise ValueError("origin must be provided when use_meters=True")
        east, north = self.deg_to_meter_simple(
            self.latitude, self.longitude, origin=origin
        )
        x_main, y_main = east, north
        x_lbl, y_lbl = "East [m]", "North [m]"
        x3_lbl, y3_lbl = "North [m]", "East [m]"
    else:
        x_main, y_main = self.longitude, self.latitude
        x_lbl, y_lbl = "Lon [deg]", "Lat [deg]"
        x3_lbl, y3_lbl = "Lat [deg]", "Lon [deg]"

    # 1) Helper to load & project CSVs
    def _load(csvs):
        dfs = []
        for path in csvs or []:
            if not os.path.exists(path):
                print(f"⚠️  {path} not found – skipped")
                continue
            df = (
                pd.read_csv(path)
                .rename(
                    columns={
                        "timestamp [unix epoch s]": "timestamp",
                        "latitude [deg]": "latitude",
                        "longitude [deg]": "longitude",
                        "depth [m]": "depth",
                        "roll [deg]": "roll",
                        "pitch [deg]": "pitch",
                        "yaw [deg]": "yaw",
                        "altitude [m]": "altitude",
                    }
                )
                .query("altitude == altitude")
            )
            if use_meters:
                e, n = self.deg_to_meter_simple(
                    df.latitude.values, df.longitude.values, origin=origin
                )
                df["x_coords"], df["y_coords"] = e, n
                df["x3_coords"], df["y3_coords"] = n, e
            else:
                df["x_coords"], df["y_coords"] = df.longitude, df.latitude
                df["x3_coords"], df["y3_coords"] = df.latitude, df.longitude
            dfs.append(df)
        return dfs

    transects = _load(transect_csv_paths)
    hilite = _load(highlighted_csv_paths or [])

    if not (transects or hilite):
        print("No valid transect files – falling back to plot_combined()")
        self.plot_combined(use_meters=use_meters, origin=origin)
        return

    # 2) Position scatter (2D)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(x_main, y_main, s=1, color="grey", alpha=0.6)
    for df in transects:
        ax.scatter(df.x_coords, df.y_coords, s=3, color="orange", alpha=0.9)
    if hilite:
        df_h = hilite[0]
        ax.scatter(
            df_h.x_coords,
            df_h.y_coords,
            s=5,
            color="red",
            alpha=1.0,
            label=highlight_label,
        )
    ax.set_xlabel(x_lbl)
    ax.set_ylabel(y_lbl)
    ax.grid(alpha=0.3)
    if hilite:
        ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

    # 3) Time-series plots
    signals = [
        ("latitude", "Lat [deg]"),
        ("longitude", "Lon [deg]"),
        ("depth", "Depth [m]"),
        ("roll", "Roll [deg]"),
        ("pitch", "Pitch [deg]"),
        ("yaw", "Yaw [deg]"),
    ]
    for attr, y_lbl in signals:
        fig, ax = plt.subplots(figsize=(12, 7))
        ax.plot(self.timestamp, getattr(self, attr), color="black", lw=0.5, alpha=0.7)
        for df in transects:
            ax.plot(df.timestamp, df[attr], color="blue", lw=1, alpha=0.9)
        if hilite:
            df_h = hilite[0]
            ax.plot(
                df_h.timestamp,
                df_h[attr],
                color="red",
                lw=2,
                alpha=1.0,
                label=highlight_label,
            )
        if attr == "depth":
            ax.invert_yaxis()
        ax.set_xlabel("Time [s] UTC")
        ax.set_ylabel(y_lbl)
        ax.grid(alpha=0.3)
        if hilite:
            ax.legend(loc="upper right")
        fig.tight_layout()
        plt.show()

    # 4) Altitude plot
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.plot(self.timestamp, self.altitude, color="black", lw=0.5, alpha=0.7)
    for df in transects:
        ax.plot(df.timestamp, df.altitude, color="blue", lw=1, alpha=0.9)
    if hilite:
        df_h = hilite[0]
        ax.plot(
            df_h.timestamp,
            df_h.altitude,
            color="red",
            lw=2,
            alpha=1.0,
            label=highlight_label,
        )
    ax.set_xlabel("Time [s] UTC")
    ax.set_ylabel("Alt [m]")
    ax.grid(alpha=0.3)
    if hilite:
        ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

    # ── 5) 3-D trajectory with heat-colored LINE (viridis reversed) ─────────
    from mpl_toolkits.mplot3d.art3d import Line3DCollection

    L = min(len(x_main), len(y_main), len(self.depth))
    X3, Y3, Z3 = y_main[:L], x_main[:L], self.depth[:L]

    fig = plt.figure(figsize=(12, 10))
    ax3 = fig.add_subplot(111, projection="3d")

    # Downsample for interactivity
    target_segments = 4000
    step = max(1, int(np.ceil(len(X3) / (target_segments + 1))))
    Xd, Yd, Zd = X3[::step], Y3[::step], Z3[::step]

    # Build colored line segments
    P = np.column_stack((Xd, Yd, Zd))
    if len(P) >= 2:
        segs = np.stack((P[:-1], P[1:]), axis=1)  # (N-1, 2, 3)
        depth_mid = 0.5 * (P[:-1, 2] + P[1:, 2])

        lc = Line3DCollection(segs, cmap="viridis_r", linewidth=4.0, alpha=0.9)
        lc.set_array(depth_mid)
        ax3.add_collection3d(lc)

        # axis limits & invert Z so larger depths at bottom
        ax3.set_xlim(P[:, 0].min(), P[:, 0].max())
        ax3.set_ylim(P[:, 1].min(), P[:, 1].max())
        zmin, zmax = P[:, 2].min(), P[:, 2].max()
        ax3.set_zlim(zmax, zmin)

        # Colorbar (remove if you only want z-axis)
        cbar = plt.colorbar(lc, ax=ax3, pad=0.1, shrink=0.6)
        cbar.set_label("Depth [m]")

    # Highlighted transect on top (also downsample)
    if hilite:
        df_h = hilite[0]
        Hx, Hy, Hz = df_h.x3_coords, df_h.y3_coords, df_h.depth
        hs = max(1, int(np.ceil(len(Hx) / (target_segments // 4))))
        ax3.plot(
            Hx[::hs],
            Hy[::hs],
            Hz[::hs],
            color="red",
            lw=5,
            alpha=1.0,
            label=highlight_label,
            zorder=10,
        )

    # View / labels (no aspect locking)
    # ax3.view_init(elev=20, azim=135)
    ax3.set_xlabel(x3_lbl)
    ax3.set_ylabel(y3_lbl)
    ax3.set_zlabel("Depth [m]")
    ax3.invert_yaxis()
    ax3.grid(alpha=0.3)
    if hilite:
        ax3.legend(loc="upper right")

    fig.tight_layout()
    plt.show()


# attach to your class
LogData.plot_with_highlighted_transects = plot_with_highlighted_transects

In [1]:
# main_csv = r"E:\mjosa_new\navigation_data\nav_data_from_logfile.csv"
# log_data = LogData(main_csv)


main_csv = r"E:\mjosa_new\navigation_data\nav_data_merged.csv"
log_data = LogData(main_csv)


# 3) Build the lists of transect CSVs
base = r"E:\mjosa_new\get_plots\nav"  # top folder
transect_csvs = [
    rf"{base}\105051\nav_data_105051_dr.csv",
    rf"{base}\112350\nav_data_112350_dr.csv",
    rf"{base}\112632\nav_data_112632_dr.csv",
    rf"{base}\115927\nav_data_115927_dr.csv",  # ← this one will be highlighted
    rf"{base}\122509\nav_data_122509_dr.csv",
    rf"{base}\125858\nav_data_125858_dr.csv",
    rf"{base}\131005\nav_data_131005_dr.csv",
]

highlighted_csvs = [
    rf"{base}\115927\nav_data_115927_dr.csv"  # only this CSV in the highlight list
]
origin_deg = (60.801146, 10.705125)

%matplotlib inline
# 4) Call the plotting method
log_data.plot_with_highlighted_transects(
    transect_csv_paths=transect_csvs,
    highlighted_csv_paths=highlighted_csvs,
    use_meters=True,  # or True with origin=(lat0, lon0)
    origin=origin_deg,  # reference point for metre conversion
    highlight_label="highlight",
)

NameError: name 'LogData' is not defined